## Self-Supervised Multi-Slice MRI Reconstruction
This notebook aims to reconstruct multiple MRI image slices only using undersampled k-space data (not training on ground truth images).

It uses a **Cosine Annealing Scheduler** to gradually tune parameters in the shape of a cosine curve (typical practice for diffusion models), after a short linear warm-up period.

## Main Goals
- Create a working implementation of a self-supervised diffusion model for MRI reconstruction.

## Overall Setup
- Load and preprocess one MRI series
- Create a undersampling masks and divide into .$M$ (main undersampling mask), $M_r$ (random sub-mask used for trainig), and $M_p$ (= $M / M_r$)
- Use unrolled denoiser model instead of U-Net Architecture so model can be trained entirely end-to-end.
- Introduce a mapper network trained to generate local and global latent variables ($w_l$ and $w_g$, resp.) that control the fine and global features in the generated images via cross-attention and instance modulation.
- Train neural network, output loss, SSIM, MSE, PSNR, and reconstructed images.

Based on the paper by Korkmaz, Patel, and Cukur: "Self-Supervised MRI Reconstruction with Unrolled Diffsion Models"(2024) https://arxiv.org/html/2306.16654v2#S3

tcia_utils nbia is the interface to the cancer imaging archive so theres available datasets we can use

In [ ]:
!pip install tcia_utils pydicom torch matplotlib numpy torchmetrics scikit-optimize

import torch
import numpy as np
import pydicom
import matplotlib.pyplot as plt
import os
from tcia_utils import nbia
import math
import random
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F
import shutil
import pandas as pd

#import sys, subprocess
#subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-optimize"])
from skopt import gp_minimize
from skopt.space import Real, Integer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 99.2 MB/s eta 0:00:00


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


# Load Data & Build a Multi-Slice Dataset

Download one full series, sort slices into anatomical order, and randomly sample slices from the middle of the scan stack.

In [ ]:
# lists candidate series so you ca pick which scan to use
data = nbia.getSeries(collection="UPENN-GBM")
if data is None:
    raise RuntimeError("UPENN-GBM returned no data.")
print(f"'UPENN-GBM' returned {len(data)} series")

for i, s in enumerate(data[:10]):
  desc = s.get("SeriesDescription")
  nimg = s.get("ImageCount")
  print(f"Index {i}: {desc} | Images: {nimg}")

if os.path.exists("./brain_mri_rebuild"):
  shutil.rmtree("./brain_mri_rebuild")
os.mkdir("./brain_mri_rebuild")

'UPENN-GBM' returned 3680 series
Index 0: AXIAL 3D GAD: Processed_CaPTk | Images: 120
Index 1: AX FLAIR: Processed_CaPTk | Images: 32
Index 2: AX T1 PRE : Processed_CaPTk | Images: 32
Index 3: AX T2 : Processed_CaPTk | Images: 32
Index 4: AX T2 : Processed_CaPTk | Images: 32
Index 5: AX T1 MPRAGE ISOTROPIC: Processed_CaPTk | Images: 192
Index 6: AXIAL FLAIR : Processed_CaPTk | Images: 60
Index 7: AX T1 PRE : Processed_CaPTk | Images: 32
Index 8: ep2d_perf 12 CC BOLUS | Images: 900
Index 9: ep2d_perf 12 CC BOLUS | Images: 900


In [ ]:
# download one full series
nbia.downloadSeries(data[5:6], number=1, path="./brain_mri_rebuild")

# collect DICOM files
dcm_files = []
for root, dirs, files in os.walk("./brain_mri_rebuild"):
  for f in files:
    if f.lower().endswith(".dcm"):
      dcm_files.append(os.path.join(root, f))

# load datasets
datasets = [pydicom.dcmread(f) for f in dcm_files]

def slice_sort_key(ds):
  if hasattr(ds, "InstanceNumber"):
    return float(ds.InstanceNumber)
  elif hasattr(ds, "ImagePositionPatient"):
    return float(ds.ImagePositionPatient[2])
  else:
    return 0.0

datasets.sort(key=slice_sort_key)

print("Number of slices in full series:", len(datasets))
print("First slice instance number:", getattr(datasets[0], "InstanceNumber", "NA"))
print("Middle slice instance number:", getattr(datasets[len(datasets)//2], "InstanceNumber", "NA"))
print("Last slice instance number:", getattr(datasets[-1], "InstanceNumber", "NA"))

Number of slices in full series: 192
First slice instance number: 1
Middle slice instance number: 97
Last slice instance number: 192


In [ ]:
# determine number of slices to randomly sample
num_slices = 96

# sample randomly from the middle 80% of slices to avoid nearly empty top/bottom slices
start = int(0.1 * len(datasets))
end = int(0.9 * len(datasets))

available_indices = list(range(start, end))
sampled_indices = sorted(random.sample(available_indices, num_slices))
datasets = [datasets[i] for i in sampled_indices]

print("Randomly sampled slice indices", sampled_indices)
print("Number of sampled slices:", len(datasets))

Randomly sampled slice indices [20, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 34, 36, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 50, 52, 53, 54, 55, 56, 59, 60, 62, 63, 64, 65, 66, 67, 69, 70, 74, 75, 76, 77, 78, 81, 84, 87, 89, 90, 92, 96, 97, 98, 99, 100, 101, 103, 104, 106, 108, 109, 111, 112, 116, 118, 121, 122, 123, 124, 126, 127, 129, 133, 134, 135, 136, 137, 139, 141, 145, 146, 147, 148, 152, 153, 156, 157, 158, 160, 161, 162, 165, 166, 169, 170, 171]
Number of sampled slices: 96


## Preprocessing Results
Each DICOM slice is converted into a normalized grayscale tensor and resized to a fixed resolution.

Output per slice: a tensor of shape 1 x 256 x 256 with values approximately in [0,1].

We sampled from the middle slices because the top and bottom slices in an MRI often contain much less structure. Sampling from the middle gives slices with more anatomical content for training and testing.

### Slice preprocessing

Converts each DICOM file into a tensor by normalizing to [0,1] and resizing to 256 x 256

In [ ]:
def dicom_to_tensor(ds, out_size=(256,256)):
  arr = ds.pixel_array.astype(np.float32)

  slope = float(getattr(ds, "RescaleSlope", 1.0))
  intercept = float(getattr(ds, "RescaleIntercept", 0.0))
  arr = arr * slope + intercept

  p_low, p_high = np.percentile(arr, (1, 99))
  arr = np.clip(arr, p_low, p_high)
  arr = (arr - p_low) / (p_high - p_low + 1e-12)

  t = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
  t = F.interpolate(t, size=out_size, mode="bilinear", align_corners=False)
  return t.squeeze(0) # [1, H, W]

target_images = torch.stack([dicom_to_tensor(ds) for ds in datasets], dim=0) # [N, 1, H, W]
N, C, H, W = target_images.shape

print("target_images shape:", target_images.shape)
print("dtype:", target_images.dtype)

target_images shape: torch.Size([96, 1, 256, 256])
dtype: torch.float32


# Zero-Filled FFT Reconstruction Setup

Splitting into train / test / validation sets

In [ ]:
# train / validation / test split over slices
num_images = target_images.shape[0]
all_indices = list(range(num_images))
rng = np.random.default_rng(seed)
rng.shuffle(all_indices)

train_end = int(0.70 * num_images)
val_end = int(0.85 * num_images)

train_indices = all_indices[:train_end]
val_indices = all_indices[train_end:val_end]
test_indices = all_indices[val_end:]

# TEMPORARY: smaller subsets so tuning finishes
train_indices = train_indices[:8]
val_indices = val_indices[:4]
test_indices = test_indices[:4]

print(f"num_images = {num_images}")
print(f"train / val / test = {len(train_indices)} / {len(val_indices)} / {len(test_indices)}")

num_images = 96
train / val / test = 8 / 4 / 4


In [ ]:
# same mask for all slices to make fair hyperparameter comparisons
def create_variable_density_mask(H, W, center_fraction=0.08, accel=4.0, seed=42):
  """
  2D variable-density mask:
  - fully samples a small low-frequency center
  - samples outer k-space with radius-dependent probability
  """
  yy, xx = torch.meshgrid(
      torch.linspace(-1.0, 1.0, H),
      torch.linspace(-1.0, 1.0, W),
      indexing="ij"
  )
  rr = torch.sqrt(xx**2 + yy **2)

  prob = 1.0 / accel + (1.0 - 1.0 / accel) * torch.exp(-6.0 * rr**2)

  center_h = max(1, int(H*center_fraction))
  center_w = max(1, int(W*center_fraction))
  h0 = H // 2 - center_h // 2
  h1 = h0 + center_h
  w0 = W // 2 - center_w // 2
  w1 = w0 + center_w

  g = torch.Generator(device="cpu")
  g.manual_seed(seed)
  random_field = torch.rand((H, W), generator=g)

  mask_real = (random_field < prob).float()
  mask_real[h0:h1, w0:w1] = 1.0
  return mask_real

mask = create_variable_density_mask(H, W, center_fraction=0.08, accel=4.0, seed=seed).to(device)
mask = mask.to(torch.complex64)

representative_test_idx = test_indices[0]
target_image = target_images[representative_test_idx, 0].to(device)

k_full = torch.fft.fftshift(torch.fft.fft2(target_image))
y_obs = k_full * mask

print(f"Representative held-out test slice: {representative_test_idx}")
print(f"k_full shape: {k_full.shape}")
print(f"mask shape: {mask.shape}")
print(f"y_obs shape: {y_obs.shape}")

def ifft2(k_space_data):
  """
  Applies 2D inverse Fast Fourier Transform and returns the real part.
  """
  return torch.fft.ifft2(k_space_data).real

Representative held-out test slice: 41
k_full shape: torch.Size([256, 256])
mask shape: torch.Size([256, 256])
y_obs shape: torch.Size([256, 256])


## Self-Supervised Diffusion Model

Step 1: Zero-Filled Reconstruction

In [ ]:
# zero-filled reconstruction
def zero_filled_reconstruction(img, mask):
  """
  img: [H, W] or [1, H, W]
  mask: [H, W]
  returns: [H, W] real zero-filled reconstruction
  """
  if img.dim() == 3:
    img2d = img[0]
  else:
    img2d = img

  k_full = torch.fft.fftshift(torch.fft.fft2(img2d), dim=(-2, -1))
  k_under = k_full * mask.to(img.device)

  x_zf = torch.fft.ifft2(torch.fft.ifftshift(k_under, dim=(-2,-1))).real
  x_zf = torch.clamp(x_zf, 0.0, 1.0)
  return x_zf

In [ ]:
zf_recon_dict = {}

for i in range(target_images.shape[0]):
  img = target_images[i].cpu()
  zf_recon_dict[i] = zero_filled_reconstruction(img, mask.cpu())

print(f"Built {len(zf_recon_dict)} zero-filled reconstructions")

Built 96 zero-filled reconstructions


Step 2: Data Preparation (split into training/test data, scaling)

In [ ]:
# train / val / test
x_train_zf = torch.stack([zf_recon_dict[i].unsqueeze(0) for i in train_indices], dim=0).float()
y_train = target_images[train_indices].cpu().float()

x_val_zf = torch.stack([zf_recon_dict[i].unsqueeze(0) for i in val_indices], dim=0).float()
y_val = target_images[val_indices].cpu().float()

x_test_zf = torch.stack([zf_recon_dict[i].unsqueeze(0) for i in test_indices], dim=0).float()
y_test = target_images[test_indices].cpu().float()

print("x_train_zf:", x_train_zf.shape, "y_train:", y_train.shape)
print("x_val_zf:", x_val_zf.shape, "y_val:", y_val.shape)
print("x_test_zf:", x_test_zf.shape, "y_test:", y_test.shape)

x_train_zf: torch.Size([8, 1, 256, 256]) y_train: torch.Size([8, 1, 256, 256])
x_val_zf: torch.Size([4, 1, 256, 256]) y_val: torch.Size([4, 1, 256, 256])
x_test_zf: torch.Size([4, 1, 256, 256]) y_test: torch.Size([4, 1, 256, 256])


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    def __init__(self, x, y, normalize=True):
        self.x = x  # Zero-filled images (x_train_zf)
        self.y = y  # Ground truth images (y_train)
        self.normalize = normalize

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx): # scaling and convert to k-space

        x_slice = self.x[idx].clone().float()
        y_slice = self.y[idx].clone().float()

        k_full = torch.fft.fftshift(torch.fft.fft2(y_slice), dim=(-2, -1))

        if self.normalize:

            v_max = torch.quantile(x_slice, 0.99)
            x_slice = torch.clamp(x_slice / (v_max + 1e-8), 0, 1)

            ty_max = torch.quantile(y_slice, 0.99)
            y_slice = torch.clamp(y_slice / (ty_max + 1e-8), 0, 1)

        return y_slice, k_full

train_dataset = PairDataset(x_train_zf, y_train)
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)

Step 3: K-space masking for self-supervision

In [ ]:
# masks with new centering and scaling
def create_ss_diff_masks(H, W, center_fraction=0.08, accel=4.0, ss_fraction=0.05, seed=42):
    # empty grid
    yy, xx = torch.meshgrid(
        torch.linspace(-1.0, 1.0, H),
        torch.linspace(-1.0, 1.0, W),
        indexing="ij"
    )
    rr = torch.sqrt(xx**2 + yy**2)
    prob = 1.0 / accel + (1.0 - 1.0 / accel) * torch.exp(-4.0 * rr**2)

    g = torch.Generator(device="cpu").manual_seed(seed)

    # base mask M
    random_field = torch.rand((H, W), generator=g)
    mask_m = (random_field < prob).float()

    # center region
    c_h, c_w = int(H * center_fraction), int(W * center_fraction)
    h0, w0 = (H // 2) - (c_h // 2), (W // 2) - (c_w // 2)

    # center = 1
    mask_m[h0:h0+c_h, w0:w0+c_w] = 1.0

    # exclude center for self-supervised loop
    center_mask = torch.zeros_like(mask_m)
    center_mask[h0:h0+c_h, w0:w0+c_w] = 1.0

    # split M into Mr and Mp
    candidate_mask = (mask_m > 0.5) & (center_mask < 0.5)
    candidate_indices = torch.where(candidate_mask)
    num_candidates = candidate_indices[0].shape[0]

    num_sampled_total = torch.sum(mask_m > 0.5).item()
    num_mr = int(num_sampled_total * ss_fraction)

    # shuffle candidates to pick Mr
    perm = torch.randperm(num_candidates, generator=g)
    mr_idx_h = candidate_indices[0][perm[:num_mr]]
    mr_idx_w = candidate_indices[1][perm[:num_mr]]

    mask_r = torch.zeros_like(mask_m)
    mask_r[mr_idx_h, mr_idx_w] = 1.0

    # Mp = M - Mr (mp keeps center)
    mask_p = mask_m - mask_r

    # reshape for (B, C, H, W) broadcasting
    return mask_m.view(1, 1, H, W), mask_r.view(1, 1, H, W), mask_p.view(1, 1, H, W)

Step 4: Data Consistency Layer

In [ ]:
# data consistency layer

class DataConsistency(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x_recon, y_p, mask_p):
        """
        Eq. 12: x_out = IFFT( FFT(x_recon) * (1 - Mp) + y_p * Mp )
        x_recon: current image estimate (batch, 1, H, W) - Complex or Real
        y_p: sampled k-space (batch, 1, H, W)
        mask_p: sampling mask (batch, 1, H, W)
        """
        # Move to k-space
        k_recon = torch.fft.fftshift(torch.fft.fft2(x_recon), dim=(-2, -1))
        mask_p = mask_p.to(k_recon.device)

        # Replace sampled points with original y_p
        k_out = k_recon * (1 - mask_p) + y_p

        # Back to image space
        x_out = torch.fft.ifft2(torch.fft.ifftshift(k_out, dim=(-2, -1)))
        return x_out.abs() # Returning magnitude for next block

Step 5: Denoising

In [ ]:
# Denoising Block

class SSDiffBlock(nn.Module):
    def __init__(self, channels=64):
        super().__init__()
        # Simplified Modulated Conv (Standard conv for backbone,
        # but you can scale weights by wg in forward)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

        # Cross-Attention (Eq. 10)
        # We use MultiheadAttention where Image=Query and Local Latent=Key/Value
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=4, batch_first=True)

        # DC Layer
        self.dc = DataConsistency()

        # Channel adapters (to move between feature space and image space)
        self.to_features = nn.Conv2d(1, channels, 1)
        self.to_image = nn.Conv2d(channels, 1, 1)

    def forward(self, x, wl, wg, y_p, mask_p):
        # 1. Modulate with wg (Global Latent)
        # Simple implementation: wg acts as a channel-wise scale
        feat = self.to_features(x)
        feat = feat * wg.view(-1, feat.shape[1], 1, 1)
        feat = self.conv1(feat)

        # 2. Cross-Attention with wl (Local Latent)
        b, c, h, w = feat.shape
        # Flatten image for attention: (batch, seq_len, channels)
        feat_flat = feat.view(b, c, h*w).permute(0, 2, 1)
        # wl is the "Context" (Key and Value)
        # Note: wl needs to be projected to match 'channels' dimension
       # context, _ = self.attn(feat_flat, wl, wl)
        context, _ = self.attn(query=feat_flat, key=wl, value=wl)
        feat = context.permute(0, 2, 1).view(b, c, h, w)

        # 3. Data Consistency (Project back to image space first)
        x_img = self.to_image(feat)
        x_out = self.dc(x_img, y_p, mask_p)

        return x_out

# mapper network explains how much denoising to apply
class MapperNetwork(nn.Module):
    def __init__(self, input_dim=2, latent_dim=32, num_layers=12):
        super().__init__()
        # input_dim=2 (e.g., normalized time 't' and 'acceleration rate')

        layers = []
        in_f = input_dim
        for _ in range(num_layers):
            layers.append(nn.Linear(in_f, latent_dim))
            layers.append(nn.ReLU())
            in_f = latent_dim

        self.net = nn.Sequential(*layers)

        # Projections to match the Unrolled Block's internal channel count (e.g., 64)
        self.to_local = nn.Linear(latent_dim, 64)  # For Cross-Attention
        self.to_global = nn.Linear(latent_dim, 64) # For Modulated Conv

    def forward(self, t, accel_rate):
        # Concatenate metadata: (Batch, 2)
        meta = torch.cat([t.unsqueeze(-1), accel_rate.unsqueeze(-1)], dim=-1)

        latent = self.net(meta) # (Batch, 32)

        wl = self.to_local(latent).unsqueeze(1) # (Batch, 1, 64) -> Key/Value for Attention
        wg = self.to_global(latent)             # (Batch, 64) -> Scale for Conv

        return wl, wg

# denoising Network
class SSDiffReconModel(nn.Module):
    def __init__(self, num_blocks=5):
        super().__init__()
        self.mapper = MapperNetwork() # 12-layer MLP from earlier
        self.blocks = nn.ModuleList([SSDiffBlock() for _ in range(num_blocks)])
        self.final_activation = nn.Sigmoid() # sigmoid function

    def forward(self, xt_up, y_p, mask_p, t, accel_rate):

        # 1. Get Latents from Mapper
        wl, wg = self.mapper(t, accel_rate)

        # 2. Iterative Unrolling
        current_x = xt_up
        for block in self.blocks:
            current_x = block(current_x, wl, wg, y_p, mask_p)

        x_hat_0 = self.final_activation(current_x)

        return x_hat_0 # This is x_hat_0 (the predicted clean image)

In [ ]:
def train_step(model, target_image, k_full, masks, optimizer, scheduler):
    """
    masks: Tuple of (M, Mr, Mp) from our previous function
    k_full: The ground truth k-space (used only to extract target_mr)
    """
    M, Mr, Mp = masks
    device = target_image.device

    # 1. Create the base undersampled image (x_u,p) using Mp
    y_p = k_full * Mp
    x_up = torch.fft.ifft2(torch.fft.ifftshift(y_p, dim=(-2, -1))).abs()

    # 2. Diffusion Forward Process (Eq. 7)
    # Pick a random time step t
    t = torch.randint(0, 1000, (1,), device=device).float() / 1000.0
    alpha_bar_t = scheduler.get_alpha_bar(t) # Standard DDPM schedule

    noise = torch.randn_like(x_up)
    # Add noise to the undersampled image, not the ground truth!
    xt_up = torch.sqrt(alpha_bar_t) * x_up + torch.sqrt(1 - alpha_bar_t) * noise

    # 3. Model Prediction (Unrolled Denoising)
    optimizer.zero_grad()

    # R_theta predicts the estimate of the fully-sampled image x0_hat
    # Note: We pass Mp so the DC layers know which points are 'real'
    x0_hat = model(xt_up, y_p, Mp, t)

    # 4. Self-Supervised Loss (Eq. 6)
    # Convert prediction to k-space
    k_hat = torch.fft.fftshift(torch.fft.fft2(x0_hat), dim=(-2, -1))

    # Compare only the points in Mr that the model NEVER saw
    # target_mr = k_full * Mr
    loss = torch.norm((k_hat * Mr) - (k_full * Mr), p=1)

    # 5. Backprop
    loss.backward()
    optimizer.step()

    return loss.item()

In [ ]:
# smapling loop

@torch.no_grad()
def reconstruct_inference(model, y_obs, mask, num_steps=5):
    """
    y_obs: Original undersampled k-space (Batch, 1, H, W)
    mask: Original mask M
    """
    model.eval()
    device = y_obs.device

    # Start with Zero-Filled Reconstruction (not Gaussian noise!)
    x_zf = torch.fft.ifft2(torch.fft.ifftshift(y_obs, dim=(-2, -1))).abs()
    xt = x_zf.clone()

    # Descending time steps (e.g., [800, 600, 400, 200, 0])
    time_steps = torch.linspace(800, 0, num_steps).to(device)

    for i in range(num_steps):
        t_val = time_steps[i] / 1000.0
        t_tensor = torch.full((y_obs.size(0),), t_val, device=device)

        # 1. Prediction: R_theta(xt, y_obs, M, t)
        # Note: We use the FULL mask M here, unlike training which used Mp
        x0_hat = model(xt, y_obs, mask, t_tensor)

        # 2. Add descending noise (Eq. 8)
        if i < num_steps - 1:
            sigma_t = 0.01 * (1 - t_val) # Example noise schedule
            z = torch.randn_like(x0_hat)
            xt = x0_hat + sigma_t * z
        else:
            xt = x0_hat # Final step, no noise

    return xt

In [ ]:
from torch.utils.data import DataLoader

# x_train (undersampled) and y_train (k-space/ground truth)
train_dataset = PairDataset(x_train_zf, y_train)

# The paper mentions a batch size of 1
dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=2
)

Blocks for visualization

In [ ]:
import matplotlib.pyplot as plt

def visualize_progress(epoch, x_up, x0_hat, target, save_path=None):
    """
    x_up: The zero-filled, undersampled input (Batch, 1, H, W)
    x0_hat: The model's current prediction (Batch, 1, H, W)
    target: The ground truth fully-sampled image (Batch, 1, H, W)
    """
    # 1. Convert to CPU and numpy
    # We use .abs() just in case complex data accidentally leaks in
    inp = x_up[0, 0].detach().cpu().abs().numpy()
    pred = x0_hat[0, 0].detach().cpu().abs().numpy()
    gt = target[0, 0].detach().cpu().abs().numpy()

    # 2. Robust Scaling: Calculate vmax from the Ground Truth
    # 99th percentile ignores extreme outliers but captures the full brain range
    vmax_val = np.percentile(gt, 99)
    vmin_val = 0 # MRI magnitude images are non-negative

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 3. Apply consistent vmin/vmax to all subplots for fair comparison
    axes[0].imshow(inp, cmap='gray', vmin=vmin_val, vmax=vmax_val)
    axes[0].set_title(f"Input (Zero-Filled)")
    axes[0].axis('off')

    axes[1].imshow(pred, cmap='gray', vmin=vmin_val, vmax=vmax_val)
    axes[1].set_title(f"SSDiffRecon Prediction")
    axes[1].axis('off')

    axes[2].imshow(gt, cmap='gray', vmin=vmin_val, vmax=vmax_val)
    axes[2].set_title("Ground Truth")
    axes[2].axis('off')

    plt.suptitle(f"Reconstruction Progress - Epoch {epoch}")

    if save_path:
        plt.savefig(save_path)

    plt.show()
    plt.close(fig) # Memory management: close figure after showing

def plot_residual(x0_hat, target):
    # Calculate the absolute difference
    residual = torch.abs(x0_hat - target)[0, 0].detach().cpu().numpy()

    plt.imshow(residual, cmap='hot') # 'hot' or 'inferno' makes errors pop
    plt.colorbar()
    plt.title("Error Residual (Prediction vs GT)")
    plt.show()

def plot_kspace_coverage(x0_hat, y_p, k_full, Mp):
    """
    x0_hat: Model prediction (Image Space)
    y_p: Input k-space (The masked version the model saw)
    k_full: The ground truth k-space
    Mp: The mask used for input
    """
    k_pred = torch.fft.fftshift(torch.fft.fft2(x0_hat), dim=(-2, -1))

    def to_log_np(tensor):
        vis = torch.log(torch.abs(tensor[0,0]) + 1e-9)
        return vis.detach().cpu().numpy()

    log_input = to_log_np(y_p)
    log_pred = to_log_np(k_pred)
    log_gt = to_log_np(k_full)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(log_input, cmap='magma')
    axes[0].set_title("Input K-Space (Masked)")

    axes[1].imshow(log_pred, cmap='magma')
    axes[1].set_title("Predicted K-Space (Interpolated)")

    axes[2].imshow(log_gt, cmap='magma')
    axes[2].set_title("Ground Truth K-Space")

    for ax in axes:
        ax.axis('off')

    plt.suptitle("K-Space Spectral Coverage")
    plt.show()

def plot_mr_accuracy(x0_hat, k_full, Mr):
    k_pred = torch.fft.fftshift(torch.fft.fft2(x0_hat), dim=(-2, -1))

    diff_at_mr = torch.abs((k_pred - k_full) * Mr)[0,0].detach().cpu().numpy()

    plt.imshow(np.log(diff_at_mr + 1e-9), cmap='inferno')
    plt.title("Error specifically at Hidden Points (Mr)")
    plt.colorbar()
    plt.show()

In [ ]:
# uses frequency-weighted loss to better capture details

def frequency_weighted_loss(k_pred, k_target, Mr, alpha=2.0):
    """
    k_pred:   (B, C, H, W) - Model Output (Likely on CUDA)
    k_target: (B, C, H, W) - Ground Truth
    Mr:       (B, C, H, W) - Loss Mask
    """
    device = k_pred.device

    k_target = k_target.to(device)
    Mr = Mr.to(device)

    B, C, H, W = k_pred.shape

    yy, xx = torch.meshgrid(
        torch.linspace(-1.0, 1.0, H, device=device),
        torch.linspace(-1.0, 1.0, W, device=device),
        indexing="ij"
    )
    W = (torch.sqrt(xx**2 + yy**2) ** alpha).to(device)

    diff = torch.abs(k_pred - k_target) * Mr
    weighted_diff = diff * W

    loss = weighted_diff.sum() / (Mr.sum() + 1e-8)

    return loss, diff, W

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from skimage.metrics import structural_similarity as ssim

# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SSDiffReconModel(num_blocks=5).to(device) # ld lr 0.002
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5, betas=(0.9, 0.999), weight_decay=1e-4)
num_epochs = 100

# start with a linear scheduler (help the model find the general image shape)
warmup_epochs = 20
warmup_scheduler = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)

# move to cosine annealer scheduler (help the model find fine details)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=(num_epochs - warmup_epochs))

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_epochs]
)

# scheduler = CosineAnnealingLR(optimizer, T_max=100) # cosine annealing only

def calculate_psnr(img1, img2, max_val=1.0):
    mse = F.mse_loss(img1, img2)
    if mse == 0: return float('inf')
    return 20 * torch.log10(max_val / torch.sqrt(mse))

# Training Loop
for epoch in range(num_epochs):
    model.train()
    epoch_ss_loss = 0
    epoch_mse = 0

    for batch_idx, (images, k_full) in enumerate(dataloader):
        images, k_full = images.to(device), k_full.to(device)
        images = (images - images.min()) / (images.max() - images.min() + 1e-8)
        k_full = torch.fft.fftshift(torch.fft.fft2(images, norm="ortho"), dim=(-2, -1))


        # generate the Self-Supervised Masks (M, Mr, Mp)
        H_dim, W_dim = images.shape[-2:]
        M, Mr, Mp = create_ss_diff_masks(H_dim, W_dim, seed=batch_idx + epoch)
        M, Mr, Mp = M.to(device), Mr.to(device), Mp.to(device)

        # undersample k-space
        y_p = k_full * Mp

        # complex reconstruction
        x_complex_zf = torch.fft.ifft2(
          torch.fft.ifftshift(y_p, dim=(-2, -1)),
          dim=(-2, -1),
          norm="ortho"
        )

        x_up = x_complex_zf.abs()
        x_up = (x_up - x_up.min()) / (x_up.max() - x_up.min() + 1e-8)
        input_phase = torch.angle(x_complex_zf)

        # forward diffusion
        t = torch.rand(images.size(0), device=device)
        a_bar = torch.cos(t * 0.5 * 3.14159)**2
        noise = torch.randn_like(x_up)
        xt_up = torch.sqrt(a_bar).view(-1,1,1,1) * x_up + torch.sqrt(1 - a_bar).view(-1,1,1,1) * noise

        accel_val = 8.0
        accel_tensor = torch.full((images.size(0),), accel_val, device=device)

        x0_hat_mag = model(xt_up, y_p, Mp, t, accel_tensor)
        x0_hat_mag = torch.clamp(x0_hat_mag, 0, 1)

        # computing k-space loss
        x0_hat_complex = x0_hat_mag * torch.exp(1j * input_phase)
        k_hat = torch.fft.fftshift(torch.fft.fft2(x0_hat_complex, norm="ortho"), dim=(-2, -1))

        # data consistency
        k_final = y_p + (k_hat * (1 - Mp))
        x0_hat_mag = torch.fft.ifft2(torch.fft.ifftshift(k_final, dim=(-2, -1)), norm="ortho").abs()

        target_mr = k_full * Mr
        ss_loss, diff, W = frequency_weighted_loss(k_final, target_mr, Mr, alpha=1.5)

        # autograd update
        if x0_hat_mag.requires_grad:
            optimizer.zero_grad()
            ss_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # log diagnostics
        with torch.no_grad():
           # troubleshooting
           # print(f"Model Output - Mean: {x0_hat_mag.mean().item():.6f} | Max: {x0_hat_mag.max().item():.6f}")
           # print(f"Input (x_up) - Mean: {x_up.mean().item():.6f} | Max: {x_up.max().item():.6f}")
            pred_np = x0_hat_mag[0, 0].cpu().numpy()
            gt_np = images[0, 0].cpu().numpy()

            current_loss_val = ss_loss.item()
            mse_val = F.mse_loss(x0_hat_mag, images).item()
            psnr_val = calculate_psnr(x0_hat_mag, images)
            ssim_val = ssim(gt_np, pred_np, data_range=1.0)

            # normalize alignment (troubleshooting)
           # p_norm = (x0_hat_mag[0] - x0_hat_mag[0].min()) / (x0_hat_mag[0].max() - x0_hat_mag[0].min() + 1e-12)
           # t_norm = (images[0] - images[0].min()) / (images[0].max() - images[0].min() + 1e-12)

            # Check for the DC component / Max Intensity Location (troubleshooting)
           # p_max_idx = torch.argmax(p_norm).item()
           # t_max_idx = torch.argmax(t_norm).item()
           # ks_max_idx = torch.argmax(torch.abs(y_p[0])).item()

            # Image Correlation
           # correlation = torch.corrcoef(torch.stack([p_norm.flatten(), t_norm.flatten()]))[0, 1].item()

            # Spectral Breakdown
            hf_mask = (W > 0.5).float()
            hf_err = (diff * hf_mask).sum() / (Mr * hf_mask).sum().clamp(min=1e-8)

            # Diagnostics printing
            if batch_idx % 10 == 0:
              print(f"Epoch {epoch} | "
                      f"SS-Loss: {current_loss_val:.5f} | MSE: {mse_val:.6f} | SSIM: {ssim_val:.4f} | PSNR: {psnr_val:.2f}dB")
              #  print(f"Batch {batch_idx} | Corr: {correlation:.4f} | K-Space DC Index: {ks_max_idx}")
               # print(f"Max Indices -> Pred: {p_max_idx}, True: {t_max_idx}")

            # visually inspect every 20 epochs
            if (epoch + 1) % 20 == 0:
              visualize_progress(epoch + 1, x_up, x0_hat_mag, images)
             # plot_residual(x0_hat_mag, images)
             # plot_kspace_coverage(x0_hat_mag, y_p, k_full, Mp)
             # plot_mr_accuracy(x0_hat_mag, k_full, Mr)

        epoch_ss_loss += current_loss_val
        epoch_mse += mse_val

    scheduler.step()
 #   print(f"--- Epoch {epoch} Complete | Avg Loss: {epoch_ss_loss/len(dataloader):.6f} ---")

Loss is *not* consistently decreasing because of the mask resampling and relatively high learning rate. This is common in self-correcting models.

Instead of training on the 'ground-truth' image, it now only has a small subset of the k-space ($M_p$) and is tested on $M_r$.

Scheduler changes the learning rate gradually to reduce the noise floor
* started using only Cosine Annealing -> had trouble finding general structure of the image
* switch to linear -> cosine annealing -> allows model to find general structure before learning rate drops